In [2]:
import pandas as pd
import sqlite3

# Load CSV into SQLite database memory
conn = sqlite3.connect(':memory:')
df = pd.read_csv("district_data_v4.csv", dtype=str)
df.to_sql("students", conn, index=False)

# Quick test: How many students?
pd.read_sql("SELECT COUNT(*) AS total_students FROM students", conn)

,total_students
0,1505


# SQL Analysis: Florida District Assessment Data

### Going deeper with SQL on the same dataset powering the Streamlit dashboard

In [3]:
# What columns do we have to work with?
pd.read_sql(""" SELECT * FROM students LIMIT 3 """, conn
            )

,LOC,STU_ID,NAME,DEUSS,ELL,ESE,ETHN,GRADE,2526 Absences,COURSE_TITLE,...,AP1.Math SS,Math.Pts4LG,MATH L25/35,2425 SCI_TEST,2425 SCI_Lvl,2425 SOC_TEST,2425 SOC_Lvl,STUDENT_FLEID,FTE,Cnt4Prof
0,6128,7423388,"Student, E561",None,N,N,H,6,4,M/J Grade 6 Accel,...,None,18,None,SCI_05,3,None,None,FL3055125183,None,None
1,6128,7550634,"Student, C868",None,N,Y,H,6,9,M/J Grade 6 Math,...,None,10,None,SCI_05,5,None,None,FL8647429897,None,None
2,6128,5304572,"Student, E317",None,N,N,H,6,2,M/J Grade 6 Math,...,None,19,None,SCI_05,2,None,None,FL6633664919,None,None


In [16]:
# Proficiency rates by school and subject in one query
pd.read_sql("""
    SELECT 
        LOC,
        COUNT(*) AS total_students,
        
        -- ELA: % with PM3 Level 3+
        ROUND(AVG(CASE WHEN CAST("FAST_ELA_PM3.Level" AS REAL) >= 3 
                       THEN 1.0 ELSE 0.0 END) * 100, 1) AS ela_proficiency,
        
        -- Math: excludes Algebra 1 (NULL skips them in AVG)
        ROUND(AVG(CASE WHEN COURSE_TITLE != 'Algebra 1 Hon' 
                        AND CAST("FAST_Math_PM3.Level" AS REAL) >= 3 THEN 1.0
                       WHEN COURSE_TITLE != 'Algebra 1 Hon' THEN 0.0
                       ELSE NULL END) * 100, 1) AS math_proficiency
        
    FROM students
    GROUP BY LOC
""", conn)

,LOC,total_students,ela_proficiency,math_proficiency
0,6128,259,88.0,90.0
1,6161,514,57.8,59.7
2,6171,732,48.4,46.4


## Learning Gains by School

In [5]:
# Learning gains: % of students who grew enough to hit their Pts4LG target
pd.read_sql("""
    SELECT 
        LOC,
        
        -- ELA: met LG if PM3 - prior year >= Pts4LG
        ROUND(AVG(CASE WHEN (CAST("FAST_ELA_PM3.SS" AS REAL) - CAST("2425 ELA_SS" AS REAL)) 
                            >= CAST("ELA Pts4LG" AS REAL)
                       THEN 1.0 ELSE 0.0 END) * 100, 1) AS ela_lg_pct,
        
        -- Math: same logic, ALL students including Algebra 1
        ROUND(AVG(CASE WHEN (CAST("FAST_Math_PM3.SS" AS REAL) - CAST("2425 MATH_SS" AS REAL)) 
                            >= CAST("Math.Pts4LG" AS REAL)
                       THEN 1.0 ELSE 0.0 END) * 100, 1) AS math_lg_pct
        
    FROM students
    WHERE "ELA Pts4LG" IS NOT NULL
    GROUP BY LOC
""", conn)

,LOC,ela_lg_pct,math_lg_pct
0,6128,72.6,71.4
1,6161,62.3,67.1
2,6171,59.8,49.7


## Subgroup Equity: ELL vs Non-ELL

In [6]:
# Subgroup comparison: ELL vs Non-ELL proficiency side by side
pd.read_sql("""
    SELECT 
        LOC,
        
        -- ELL ELA proficiency
        ROUND(AVG(CASE WHEN ELL = 'Y' AND CAST("FAST_ELA_PM3.Level" AS REAL) >= 3 
                       THEN 1.0
                       WHEN ELL = 'Y' THEN 0.0 
                       ELSE NULL END) * 100, 1) AS ell_ela_prof,
        
        -- Non-ELL ELA proficiency
        ROUND(AVG(CASE WHEN ELL = 'N' AND CAST("FAST_ELA_PM3.Level" AS REAL) >= 3 
                       THEN 1.0
                       WHEN ELL = 'N' THEN 0.0 
                       ELSE NULL END) * 100, 1) AS non_ell_ela_prof,
        
        -- Gap
        ROUND(AVG(CASE WHEN ELL = 'N' AND CAST("FAST_ELA_PM3.Level" AS REAL) >= 3 
                       THEN 1.0
                       WHEN ELL = 'N' THEN 0.0 
                       ELSE NULL END) * 100, 1)
        - ROUND(AVG(CASE WHEN ELL = 'Y' AND CAST("FAST_ELA_PM3.Level" AS REAL) >= 3 
                       THEN 1.0
                       WHEN ELL = 'Y' THEN 0.0 
                       ELSE NULL END) * 100, 1) AS ela_gap
        
    FROM students
    GROUP BY LOC
""", conn)

,LOC,ell_ela_prof,non_ell_ela_prof,ela_gap
0,6128,75.0,90.4,15.4
1,6161,48.8,62.0,13.2
2,6171,36.5,57.5,21.0


## Bubble Students: Within 10 Points of Proficiency (using CTEs)

In [17]:
# CTE: bubble students based on PM2 scores (actionable before PM3)
pd.read_sql("""
    WITH proficiency_cuts AS (
        SELECT 6 AS grade, 225 AS ela_cut, 229 AS math_cut
        UNION ALL SELECT 7, 232, 235
        UNION ALL SELECT 8, 238, 244
    ),
    student_gaps AS (
        SELECT 
            s.LOC,
            s.GRADE,
            CAST(s."FAST_ELA_PM2.SS" AS REAL) - p.ela_cut AS ela_distance,
            CAST(s."FAST_Math_PM2.SS" AS REAL) - p.math_cut AS math_distance
        FROM students s
        JOIN proficiency_cuts p ON CAST(s.GRADE AS INT) = p.grade
        WHERE s.COURSE_TITLE != 'Algebra 1 Hon'
    )
    SELECT 
        LOC,
        COUNT(*) AS total_non_alg1,
        SUM(CASE WHEN ela_distance BETWEEN -10 AND -1 THEN 1 ELSE 0 END) AS ela_bubble_pm2,
        SUM(CASE WHEN math_distance BETWEEN -10 AND -1 THEN 1 ELSE 0 END) AS math_bubble_pm2
    FROM student_gaps
    GROUP BY LOC
""", conn)

,LOC,total_non_alg1,ela_bubble_pm2,math_bubble_pm2
0,6128,160,31,28
1,6161,273,46,52
2,6171,576,90,135


In [18]:
# CTE: bubble students based on prior year (2425) scores
pd.read_sql("""
    WITH proficiency_cuts AS (
        SELECT 6 AS grade, 225 AS ela_cut, 229 AS math_cut
        UNION ALL SELECT 7, 232, 235
        UNION ALL SELECT 8, 238, 244
    ),
    student_gaps AS (
        SELECT 
            s.LOC,
            s.GRADE,
            CAST(s."2425 ELA_SS" AS REAL) - p.ela_cut AS ela_distance,
            CAST(s."2425 MATH_SS" AS REAL) - p.math_cut AS math_distance
        FROM students s
        JOIN proficiency_cuts p ON CAST(s.GRADE AS INT) = p.grade
        WHERE s.COURSE_TITLE != 'Algebra 1 Hon'
    )
    SELECT 
        LOC,
        COUNT(*) AS total_non_alg1,
        SUM(CASE WHEN ela_distance BETWEEN -10 AND -1 THEN 1 ELSE 0 END) AS ela_bubble_2425,
        SUM(CASE WHEN math_distance BETWEEN -10 AND -1 THEN 1 ELSE 0 END) AS math_bubble_2425
    FROM student_gaps
    GROUP BY LOC
""", conn)

,LOC,total_non_alg1,ela_bubble_2425,math_bubble_2425
0,6128,160,27,35
1,6161,273,28,64
2,6171,576,86,111


## Bubble Student Outcomes: Did They Cross the Line?

In [19]:
# Of PM2 bubble students, how many reached proficiency by PM3?
pd.read_sql("""
    WITH proficiency_cuts AS (
        SELECT 6 AS grade, 225 AS ela_cut, 229 AS math_cut
        UNION ALL SELECT 7, 232, 235
        UNION ALL SELECT 8, 238, 244
    ),
    ela_bubble AS (
        SELECT s.LOC, s.GRADE,
            CAST(s."FAST_ELA_PM2.SS" AS REAL) AS pm2,
            CAST(s."FAST_ELA_PM3.SS" AS REAL) AS pm3,
            p.ela_cut AS cut
        FROM students s
        JOIN proficiency_cuts p ON CAST(s.GRADE AS INT) = p.grade
        WHERE CAST(s."FAST_ELA_PM2.SS" AS REAL) BETWEEN (p.ela_cut - 10) AND (p.ela_cut - 1)
    )
    SELECT 
        LOC,
        COUNT(*) AS ela_bubble_total,
        SUM(CASE WHEN pm3 >= cut THEN 1 ELSE 0 END) AS reached_proficiency,
        SUM(CASE WHEN pm3 < pm2 THEN 1 ELSE 0 END) AS dropped,
        SUM(CASE WHEN pm3 >= pm2 AND pm3 < cut THEN 1 ELSE 0 END) AS grew_but_missed
    FROM ela_bubble
    GROUP BY LOC
""", conn)

,LOC,ela_bubble_total,reached_proficiency,dropped,grew_but_missed
0,6128,43,40,0,3
1,6161,61,0,59,2
2,6171,107,1,105,1


In [20]:
# Math PM2 bubble outcomes
pd.read_sql("""
    WITH proficiency_cuts AS (
        SELECT 6 AS grade, 225 AS ela_cut, 229 AS math_cut
        UNION ALL SELECT 7, 232, 235
        UNION ALL SELECT 8, 238, 244
    ),
    math_bubble AS (
        SELECT s.LOC, s.GRADE,
            CAST(s."FAST_Math_PM2.SS" AS REAL) AS pm2,
            CAST(s."FAST_Math_PM3.SS" AS REAL) AS pm3,
            p.math_cut AS cut
        FROM students s
        JOIN proficiency_cuts p ON CAST(s.GRADE AS INT) = p.grade
        WHERE s.COURSE_TITLE != 'Algebra 1 Hon'
          AND CAST(s."FAST_Math_PM2.SS" AS REAL) BETWEEN (p.math_cut - 10) AND (p.math_cut - 1)
    )
    SELECT 
        LOC,
        COUNT(*) AS math_bubble_total,
        SUM(CASE WHEN pm3 >= cut THEN 1 ELSE 0 END) AS reached_proficiency,
        SUM(CASE WHEN pm3 < pm2 THEN 1 ELSE 0 END) AS dropped,
        SUM(CASE WHEN pm3 >= pm2 AND pm3 < cut THEN 1 ELSE 0 END) AS grew_but_missed
    FROM math_bubble
    GROUP BY LOC
""", conn)

,LOC,math_bubble_total,reached_proficiency,dropped,grew_but_missed
0,6128,28,23,3,2
1,6161,52,8,41,3
2,6171,135,43,81,11


## Bubble Student Visualizations

In [24]:
import plotly.express as px
import plotly.subplots as make_subplots

# Prior year (2425) bubble students by school
prior_bubble = pd.read_sql("""
    WITH proficiency_cuts AS (
        SELECT 6 AS grade, 225 AS ela_cut, 229 AS math_cut
        UNION ALL SELECT 7, 232, 235
        UNION ALL SELECT 8, 238, 244
    ),
    student_gaps AS (
        SELECT s.LOC,
            CAST(s."2425 ELA_SS" AS REAL) - p.ela_cut AS ela_dist,
            CAST(s."2425 MATH_SS" AS REAL) - p.math_cut AS math_dist
        FROM students s
        JOIN proficiency_cuts p ON CAST(s.GRADE AS INT) = p.grade
        WHERE s.COURSE_TITLE != 'Algebra 1 Hon'
    )
    SELECT LOC,
        SUM(CASE WHEN ela_dist BETWEEN -10 AND -1 THEN 1 ELSE 0 END) AS ELA,
        SUM(CASE WHEN math_dist BETWEEN -10 AND -1 THEN 1 ELSE 0 END) AS Math
    FROM student_gaps GROUP BY LOC
""", conn)
prior_bubble["LOC"] = prior_bubble["LOC"].astype(str)
prior_long = prior_bubble.melt(id_vars="LOC", var_name="Subject", value_name="Students")

fig = px.bar(prior_long, x="LOC", y="Students", color="Subject",
             barmode="group", title="Prior Year (2425) Bubble Students: Within 10 Points of Proficiency")
fig.show()

In [25]:
# PM2 bubble students by school
pm2_bubble = pd.read_sql("""
    WITH proficiency_cuts AS (
        SELECT 6 AS grade, 225 AS ela_cut, 229 AS math_cut
        UNION ALL SELECT 7, 232, 235
        UNION ALL SELECT 8, 238, 244
    ),
    student_gaps AS (
        SELECT s.LOC,
            CAST(s."FAST_ELA_PM2.SS" AS REAL) - p.ela_cut AS ela_dist,
            CAST(s."FAST_Math_PM2.SS" AS REAL) - p.math_cut AS math_dist
        FROM students s
        JOIN proficiency_cuts p ON CAST(s.GRADE AS INT) = p.grade
        WHERE s.COURSE_TITLE != 'Algebra 1 Hon'
    )
    SELECT LOC,
        SUM(CASE WHEN ela_dist BETWEEN -10 AND -1 THEN 1 ELSE 0 END) AS ELA,
        SUM(CASE WHEN math_dist BETWEEN -10 AND -1 THEN 1 ELSE 0 END) AS Math
    FROM student_gaps GROUP BY LOC
""", conn)
pm2_bubble["LOC"] = pm2_bubble["LOC"].astype(str)
pm2_long = pm2_bubble.melt(id_vars="LOC", var_name="Subject", value_name="Students")

fig = px.bar(pm2_long, x="LOC", y="Students", color="Subject",
             barmode="group", title="PM2 Bubble Students: Within 10 Points of Proficiency")
fig.show()

In [26]:
# ELA bubble outcomes as percentage (shows school effectiveness)
ela_outcomes = pd.read_sql("""
    WITH proficiency_cuts AS (
        SELECT 6 AS grade, 225 AS ela_cut
        UNION ALL SELECT 7, 232
        UNION ALL SELECT 8, 238
    ),
    ela_bubble AS (
        SELECT s.LOC,
            CAST(s."FAST_ELA_PM2.SS" AS REAL) AS pm2,
            CAST(s."FAST_ELA_PM3.SS" AS REAL) AS pm3,
            p.ela_cut AS cut
        FROM students s
        JOIN proficiency_cuts p ON CAST(s.GRADE AS INT) = p.grade
        WHERE CAST(s."FAST_ELA_PM2.SS" AS REAL) BETWEEN (p.ela_cut - 10) AND (p.ela_cut - 1)
    )
    SELECT LOC,
        ROUND(SUM(CASE WHEN pm3 >= cut THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS "Reached Proficiency",
        ROUND(SUM(CASE WHEN pm3 >= pm2 AND pm3 < cut THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS "Grew But Missed",
        ROUND(SUM(CASE WHEN pm3 < pm2 THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS "Dropped"
    FROM ela_bubble GROUP BY LOC
""", conn)
ela_outcomes["LOC"] = ela_outcomes["LOC"].astype(str)
ela_long = ela_outcomes.melt(id_vars="LOC", var_name="Outcome", value_name="Percent")

fig = px.bar(ela_long, x="LOC", y="Percent", color="Outcome",
             barmode="stack", title="ELA Bubble Student Outcomes (% of PM2 Bubble Group)",
             color_discrete_map={"Reached Proficiency": "#6B8F71", 
                                 "Grew But Missed": "#D4A574", 
                                 "Dropped": "#B8542A"})
fig.update_layout(yaxis_title="Percentage", yaxis_range=[0, 100])
fig.show()

In [27]:
# Math bubble outcomes as percentage
math_outcomes = pd.read_sql("""
    WITH proficiency_cuts AS (
        SELECT 6 AS grade, 229 AS math_cut
        UNION ALL SELECT 7, 235
        UNION ALL SELECT 8, 244
    ),
    math_bubble AS (
        SELECT s.LOC,
            CAST(s."FAST_Math_PM2.SS" AS REAL) AS pm2,
            CAST(s."FAST_Math_PM3.SS" AS REAL) AS pm3,
            p.math_cut AS cut
        FROM students s
        JOIN proficiency_cuts p ON CAST(s.GRADE AS INT) = p.grade
        WHERE s.COURSE_TITLE != 'Algebra 1 Hon'
          AND CAST(s."FAST_Math_PM2.SS" AS REAL) BETWEEN (p.math_cut - 10) AND (p.math_cut - 1)
    )
    SELECT LOC,
        ROUND(SUM(CASE WHEN pm3 >= cut THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS "Reached Proficiency",
        ROUND(SUM(CASE WHEN pm3 >= pm2 AND pm3 < cut THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS "Grew But Missed",
        ROUND(SUM(CASE WHEN pm3 < pm2 THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS "Dropped"
    FROM math_bubble GROUP BY LOC
""", conn)
math_outcomes["LOC"] = math_outcomes["LOC"].astype(str)
math_long = math_outcomes.melt(id_vars="LOC", var_name="Outcome", value_name="Percent")

fig = px.bar(math_long, x="LOC", y="Percent", color="Outcome",
             barmode="stack", title="Math Bubble Student Outcomes (% of PM2 Bubble Group)",
             color_discrete_map={"Reached Proficiency": "#6B8F71", 
                                 "Grew But Missed": "#D4A574", 
                                 "Dropped": "#B8542A"})
fig.update_layout(yaxis_title="Percentage", yaxis_range=[0, 100])
fig.show()

## Window Functions: Student Rank Within School

In [29]:
# Window function: RANK() assigns position within each school
pd.read_sql("""
    SELECT 
        LOC,
        NAME,
        GRADE,
        CAST("FAST_ELA_PM3.SS" AS REAL) AS ela_pm3,
        RANK() OVER (PARTITION BY LOC ORDER BY CAST("FAST_ELA_PM3.SS" AS REAL) DESC) AS ela_rank,
        CAST("FAST_Math_PM3.SS" AS REAL) AS math_pm3,
        RANK() OVER (PARTITION BY LOC ORDER BY CAST("FAST_Math_PM3.SS" AS REAL) DESC) AS math_rank
    FROM students
    WHERE COURSE_TITLE != 'Algebra 1 Hon'
    ORDER BY LOC, ela_rank
    LIMIT 9
""", conn)

,LOC,NAME,GRADE,ela_pm3,ela_rank,math_pm3,math_rank
0,6128,"Student, F705",8,284.0,1,278.0,16
1,6128,"Student, C205",8,281.0,2,261.0,74
2,6128,"Student, X613",8,278.0,3,272.0,33
3,6128,"Student, J815",8,275.0,4,263.0,63
4,6128,"Student, D502",6,274.0,5,275.0,24
5,6128,"Student, X930",8,274.0,5,273.0,30
6,6128,"Student, M756",7,274.0,5,270.0,41
7,6128,"Student, J521",8,274.0,5,266.0,54
8,6128,"Student, M790",6,271.0,9,279.0,13


In [9]:
# Average score trajectory across testing windows
pd.read_sql("""
    SELECT 
        LOC,
        ROUND(AVG(CAST("FAST_ELA_PM1.SS" AS REAL)), 1) AS ela_pm1_avg,
        ROUND(AVG(CAST("FAST_ELA_PM2.SS" AS REAL)), 1) AS ela_pm2_avg,
        ROUND(AVG(CAST("FAST_ELA_PM3.SS" AS REAL)), 1) AS ela_pm3_avg,
        ROUND(AVG(CAST("FAST_ELA_PM3.SS" AS REAL)) 
            - AVG(CAST("FAST_ELA_PM1.SS" AS REAL)), 1) AS ela_total_growth,
        ROUND(AVG(CAST("FAST_Math_PM1.SS" AS REAL)), 1) AS math_pm1_avg,
        ROUND(AVG(CAST("FAST_Math_PM2.SS" AS REAL)), 1) AS math_pm2_avg,
        ROUND(AVG(CAST("FAST_Math_PM3.SS" AS REAL)), 1) AS math_pm3_avg,
        ROUND(AVG(CAST("FAST_Math_PM3.SS" AS REAL)) 
            - AVG(CAST("FAST_Math_PM1.SS" AS REAL)), 1) AS math_total_growth
    FROM students
    GROUP BY LOC
""", conn)

,LOC,ela_pm1_avg,ela_pm2_avg,ela_pm3_avg,ela_total_growth,math_pm1_avg,math_pm2_avg,math_pm3_avg,math_total_growth
0,6128,241.8,246.2,251.2,9.4,240.2,243.5,320.5,80.3
1,6161,238.3,242.9,234.0,-4.3,239.3,242.7,321.5,82.2
2,6171,231.4,236.1,228.3,-3.0,227.3,231.0,268.1,40.8


## PM1 to PM3 Growth Visualization

In [10]:
import plotly.express as px

# Pull the trajectory data
growth = pd.read_sql("""
    SELECT 
        LOC,
        ROUND(AVG(CAST("FAST_ELA_PM1.SS" AS REAL)), 1) AS PM1,
        ROUND(AVG(CAST("FAST_ELA_PM2.SS" AS REAL)), 1) AS PM2,
        ROUND(AVG(CAST("FAST_ELA_PM3.SS" AS REAL)), 1) AS PM3
    FROM students
    GROUP BY LOC
""", conn)

# Reshape from wide to long for Plotly
growth_long = growth.melt(id_vars="LOC", var_name="Window", value_name="Avg_Score")
growth_long["LOC"] = growth_long["LOC"].astype(str)

fig = px.line(growth_long, x="Window", y="Avg_Score", color="LOC",
              markers=True, title="ELA Score Trajectory: PM1 → PM3")
fig.show()

## Summary: School Grades Components Side by Side

In [13]:
# All 7 FLDOE School Grades components in one query
pd.read_sql("""
    SELECT 
        LOC,
        -- ELA Achievement
        ROUND(AVG(CASE WHEN CAST("FAST_ELA_PM3.Level" AS REAL) >= 3 
                       THEN 1.0 ELSE 0.0 END) * 100, 1) AS ela_ach,
        -- ELA Learning Gains
        ROUND(AVG(CASE WHEN (CAST("FAST_ELA_PM3.SS" AS REAL) - CAST("2425 ELA_SS" AS REAL)) 
                            >= CAST("ELA Pts4LG" AS REAL)
                       THEN 1.0 ELSE 0.0 END) * 100, 1) AS ela_lg,
        -- ELA LG Lowest 25%
        ROUND(AVG(CASE WHEN "ELA L25/35" = 'L25' 
                        AND (CAST("FAST_ELA_PM3.SS" AS REAL) - CAST("2425 ELA_SS" AS REAL)) 
                            >= CAST("ELA Pts4LG" AS REAL)
                       THEN 1.0
                       WHEN "ELA L25/35" = 'L25' THEN 0.0
                       ELSE NULL END) * 100, 1) AS ela_lg_l25,
        -- Math Achievement (excludes Algebra 1)
        ROUND(
            AVG(CASE WHEN COURSE_TITLE != 'Algebra 1 Hon' 
                      AND CAST("FAST_Math_PM3.Level" AS REAL) >= 3 
                      THEN 1.0 ELSE 0.0 END) 
            / AVG(CASE WHEN COURSE_TITLE != 'Algebra 1 Hon' 
                       THEN 1.0 ELSE NULL END) * 100, 1
        ) AS math_ach,
        -- Math Learning Gains (includes Algebra 1)
        ROUND(AVG(CASE WHEN (CAST("FAST_Math_PM3.SS" AS REAL) - CAST("2425 MATH_SS" AS REAL)) 
                            >= CAST("Math.Pts4LG" AS REAL)
                       THEN 1.0 ELSE 0.0 END) * 100, 1) AS math_lg,
        -- Math LG Lowest 25%
        ROUND(AVG(CASE WHEN "MATH L25/35" = 'L25' 
                        AND (CAST("FAST_Math_PM3.SS" AS REAL) - CAST("2425 MATH_SS" AS REAL)) 
                            >= CAST("Math.Pts4LG" AS REAL)
                       THEN 1.0
                       WHEN "MATH L25/35" = 'L25' THEN 0.0
                       ELSE NULL END) * 100, 1) AS math_lg_l25,
        -- MS Acceleration (Algebra 1 students with Level 3+)
        ROUND(AVG(CASE WHEN COURSE_TITLE = 'Algebra 1 Hon' 
                        AND CAST("FAST_Math_PM3.Level" AS REAL) >= 3
                       THEN 1.0
                       WHEN COURSE_TITLE = 'Algebra 1 Hon' THEN 0.0
                       ELSE NULL END) * 100, 1) AS ms_accel
    FROM students
    GROUP BY LOC
""", conn)

,LOC,ela_ach,ela_lg,ela_lg_l25,math_ach,math_lg,math_lg_l25,ms_accel
0,6128,88.0,72.6,75.0,55.6,71.4,66.7,87.9
1,6161,57.8,62.3,50.0,31.7,67.1,60.0,77.2
2,6171,48.4,59.8,55.9,36.5,49.7,53.6,61.5


In [30]:
import plotly.express as px

# All 7 FLDOE School Grades components in one query
summary = pd.read_sql("""
    SELECT 
        LOC,
        ROUND(AVG(CASE WHEN CAST("FAST_ELA_PM3.Level" AS REAL) >= 3 
                       THEN 1.0 ELSE 0.0 END) * 100, 1) AS "ELA Achievement",
        ROUND(AVG(CASE WHEN (CAST("FAST_ELA_PM3.SS" AS REAL) - CAST("2425 ELA_SS" AS REAL)) 
                            >= CAST("ELA Pts4LG" AS REAL)
                       THEN 1.0 ELSE 0.0 END) * 100, 1) AS "ELA Learning Gains",
        ROUND(AVG(CASE WHEN "ELA L25/35" = 'L25' 
                        AND (CAST("FAST_ELA_PM3.SS" AS REAL) - CAST("2425 ELA_SS" AS REAL)) 
                            >= CAST("ELA Pts4LG" AS REAL)
                       THEN 1.0
                       WHEN "ELA L25/35" = 'L25' THEN 0.0
                       ELSE NULL END) * 100, 1) AS "ELA LG L25",
        ROUND(AVG(CASE WHEN COURSE_TITLE != 'Algebra 1 Hon' 
                        AND CAST("FAST_Math_PM3.Level" AS REAL) >= 3 THEN 1.0
                       WHEN COURSE_TITLE != 'Algebra 1 Hon' THEN 0.0
                       ELSE NULL END) * 100, 1) AS "Math Achievement",
        ROUND(AVG(CASE WHEN (CAST("FAST_Math_PM3.SS" AS REAL) - CAST("2425 MATH_SS" AS REAL)) 
                            >= CAST("Math.Pts4LG" AS REAL)
                       THEN 1.0 ELSE 0.0 END) * 100, 1) AS "Math Learning Gains",
        ROUND(AVG(CASE WHEN "MATH L25/35" = 'L25' 
                        AND (CAST("FAST_Math_PM3.SS" AS REAL) - CAST("2425 MATH_SS" AS REAL)) 
                            >= CAST("Math.Pts4LG" AS REAL)
                       THEN 1.0
                       WHEN "MATH L25/35" = 'L25' THEN 0.0
                       ELSE NULL END) * 100, 1) AS "Math LG L25",
        ROUND(AVG(CASE WHEN COURSE_TITLE = 'Algebra 1 Hon' 
                        AND CAST("FAST_Math_PM3.Level" AS REAL) >= 3
                       THEN 1.0
                       WHEN COURSE_TITLE = 'Algebra 1 Hon' THEN 0.0
                       ELSE NULL END) * 100, 1) AS "MS Acceleration"
    FROM students
    GROUP BY LOC
""", conn)

summary["LOC"] = summary["LOC"].astype(str)
summary_long = summary.melt(id_vars="LOC", var_name="Component", value_name="Percentage")

fig = px.bar(summary_long, x="Component", y="Percentage", color="LOC",
             barmode="group", title="Miami-Dade County Public School Grades Components by School")
fig.update_layout(xaxis_tickangle=-30)
fig.show()